# Oil Prices in 2026 — Act 5: What If the Model Could Read the News?

The [companion notebook](energy_oil_case_study.ipynb) showed that Prophet's rolling 30-day
forecast catastrophically missed the 2026 oil price surge — forecasting ~$61/bbl
while WTI hit $100. The model wasn't wrong in principle; it simply had no mechanism
for incorporating the geopolitical context that was publicly available at the time.

This notebook asks: **could a context-aware LLM forecaster have done better?**

We evaluate three key forecast origins in early 2026 using three methods side by side:
- **Prophet** (baseline — already computed, loaded from cache)
- **LLMP — no context** (Gemini 3 Flash, history only)
- **LLMP — with context** (same model + plausibly-knowable geopolitical context at each origin)

And we frame the comparison three ways:

| | Question type | Evaluation |
|---|---|---|
| **Act 5** | *Trajectory* — what will the 30-day price path look like? | MAE vs. actuals |
| **Act 6** | *Binary* — will price exceed a meaningful threshold in 30 days? | Calibration of P(exceed) |
| **Act 7** | *Causal* — what forces are the model anchoring on? | Qualitative reasoning audit |

In [1]:
from __future__ import annotations

import json
import logging
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.subplots as psp
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
logging.getLogger("prophet").setLevel(logging.ERROR)

# ── Repo root: walk up from CWD until pyproject.toml is found ─────────────────
_cwd = Path(os.getcwd()).resolve()
REPO_ROOT = _cwd
while not (REPO_ROOT / "pyproject.toml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        REPO_ROOT = _cwd
        break
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"

for p in [str(REPO_ROOT / "implementations"), str(REPO_ROOT / "aieng-forecasting")]:
    if p not in sys.path:
        sys.path.insert(0, p)

load_dotenv(REPO_ROOT / ".env")

# ── Colour palette (matches companion notebook) ────────────────────────────────
CLR_HISTORY   = "#bdd7e7"
CLR_ACTUAL    = "#2171b5"   # solid blue — the truth
CLR_PROPHET   = "#636363"   # grey — the blind baseline
CLR_LLMP_BARE = "#fd8d3c"   # orange — LLM, history only
CLR_LLMP_CTX  = "#2ca02c"   # green — LLM, with context
CLR_CONFLICT  = "#d62728"   # red — conflict annotation
CONFLICT_DATE = pd.Timestamp("2026-03-01")

print(f"Repo root : {REPO_ROOT}")
print(f"Data dir  : {DATA_DIR}")
print("Setup complete.")

Repo root : /Users/ethanjackson/agentic-forecasting
Data dir  : /Users/ethanjackson/agentic-forecasting/data
Setup complete.


In [2]:
# ── WTI price history (same cache as companion notebook) ──────────────────────
PRICE_CACHE        = DATA_DIR / "wti_price_history.parquet"
PROPHET_CACHE      = DATA_DIR / "energy_case_study_forecasts_30d_daily_v3.parquet"
PROPHET_TRAJ_CACHE = DATA_DIR / "energy_prophet_trajectories.parquet"
LLMP_CACHE         = DATA_DIR / "energy_llmp_context_forecasts.parquet"

price_df = pd.read_parquet(PRICE_CACHE)
price_df.index = pd.DatetimeIndex(
    [pd.Timestamp(str(d)[:10]) for d in price_df.index]
)
price_df.index.name = "date"
price_df = price_df.sort_index()

prophet_df = pd.read_parquet(PROPHET_CACHE)
prophet_df["sim_day"]        = pd.to_datetime(prophet_df["sim_day"])
prophet_df["resolution_date"] = pd.to_datetime(prophet_df["resolution_date"])

print(f"WTI price history : {price_df.index[0].date()} → {price_df.index[-1].date()} ({len(price_df):,} days)")
print(f"Prophet forecasts : {prophet_df['sim_day'].min().date()} → {prophet_df['sim_day'].max().date()} ({len(prophet_df):,} rows)")

WTI price history : 2021-01-04 → 2026-05-01 (1,340 days)
Prophet forecasts : 2025-01-02 → 2026-04-01 (314 rows)


---

## The Setup

We pick **three forecast origins** in early 2026 — each representing a different
stage of the geopolitical escalation that drove WTI from ~$58 to $100+ between
January and April 2026.

| Origin | WTI at origin | Resolution date | Actual WTI at resolution | Prophet forecast | Context available |
|---|---|---|---|---|---|
| Jan 5, 2026 | $58 | Feb 4, 2026 | **$65** | $58 (inside CI) | Tensions building; OPEC+ cuts; insurance premiums rising |
| Feb 2, 2026 | $62 | Mar 4, 2026 | **$75** | $61 (miss — above CI) | Gulf of Oman incident; escalation fears; analyst upgrades |
| Mar 2, 2026 | $71 | Apr 1, 2026 | **$100** | $61 (catastrophic miss) | Conflict active; Strait of Hormuz blockade; IEA emergency session |

For each origin we ask: does an LLMP with access to publicly-available context
shift its forecast in the right direction — toward the actual outcome?

In [3]:
def compress_history(
    price_df: pd.DataFrame,
    as_of: pd.Timestamp,
    recent_window_months: int = 6,
) -> pd.DataFrame:
    """Return a token-efficient history: weekly averages for older data, daily for recent.

    Compresses ~1300 daily rows to ~200 rows while preserving the recent
    daily granularity that matters most for the LLM's short-horizon forecast.
    """
    hist = price_df[price_df.index <= as_of].copy()
    cutoff_daily = as_of - pd.DateOffset(months=recent_window_months)

    older = (
        hist[hist.index < cutoff_daily]
        .resample("W")
        .mean()
        .reset_index()
        .rename(columns={"date": "timestamp", "price": "value"})
    )
    recent = (
        hist[hist.index >= cutoff_daily]
        .reset_index()
        .rename(columns={"date": "timestamp", "price": "value"})
    )

    result = pd.concat([older, recent], ignore_index=True)
    result["timestamp"] = pd.to_datetime(result["timestamp"])
    return result[["timestamp", "value"]].sort_values("timestamp").reset_index(drop=True)


def prophet_row_at_origin(prophet_df: pd.DataFrame, origin: pd.Timestamp) -> pd.Series:
    """Return the Prophet forecast row whose sim_day is nearest to (on or after) origin."""
    candidates = prophet_df[prophet_df["sim_day"] >= origin]
    return candidates.iloc[0]


def resolution_price(price_df: pd.DataFrame, origin: pd.Timestamp, horizon_calendar_days: int = 30) -> tuple[pd.Timestamp, float]:
    """Return (resolution_date, actual_price) for a calendar-day horizon from origin."""
    target = origin + pd.Timedelta(days=horizon_calendar_days)
    row = price_df[price_df.index >= target].iloc[0]
    return row.name, float(row["price"])


# Sanity-check the three origins
ORIGINS = [
    pd.Timestamp("2026-01-05"),
    pd.Timestamp("2026-02-02"),
    pd.Timestamp("2026-03-02"),
]

# ── Shock experiment: 8 weekly origins spanning Feb–Mar 2026 ─────────────────
# 4 calm weeks followed by 4 shock weeks — a natural stress test.
SHOCK_ORIGINS = [
    pd.Timestamp("2026-02-02"),
    pd.Timestamp("2026-02-09"),
    pd.Timestamp("2026-02-17"),   # Tue (Mon = Presidents' Day holiday)
    pd.Timestamp("2026-02-23"),
    pd.Timestamp("2026-03-02"),
    pd.Timestamp("2026-03-09"),
    pd.Timestamp("2026-03-16"),
    pd.Timestamp("2026-03-23"),
]

print("Origin summary:")
for o in ORIGINS:
    price_at_origin = float(price_df[price_df.index >= o].iloc[0]["price"])
    res_date, res_price = resolution_price(price_df, o)
    p_row = prophet_row_at_origin(prophet_df, o)
    print(
        f"  {o.date()}  WTI=${price_at_origin:.2f}  "
        f"→ resolution {res_date.date()} actual=${res_price:.2f}  "
        f"prophet=${p_row['yhat']:.2f} [{p_row['yhat_lower']:.1f},{p_row['yhat_upper']:.1f}]  "
        f"inside_ci={p_row['inside_ci']}"
    )

Origin summary:
  2026-01-05  WTI=$58.32  → resolution 2026-02-04 actual=$65.14  prophet=$57.55 [49.5,65.5]  inside_ci=True
  2026-02-02  WTI=$62.14  → resolution 2026-03-04 actual=$74.66  prophet=$60.91 [52.8,69.4]  inside_ci=False
  2026-03-02  WTI=$71.23  → resolution 2026-04-01 actual=$100.12  prophet=$61.32 [53.3,69.3]  inside_ci=False


In [4]:
# ── Prophet full-trajectory forecasts for the three origins ──────────────────
#
# The existing Prophet cache stores only the terminal 30-calendar-day-ahead
# point per origin.  Here we re-run Prophet for just the three key origins and
# produce a 21-business-day trajectory fan — matching the LLMP output structure
# so the trajectory chart shows a like-for-like comparison between methods.
#
# Fitting 3 Prophet models takes ~10 s; results are cached to
# data/energy_prophet_trajectories.parquet so subsequent runs are instant.

from prophet import Prophet  # type: ignore[import-untyped]


def _fit_prophet_at_origin(price_df: pd.DataFrame, origin: pd.Timestamp) -> pd.DataFrame:
    """Fit one Prophet model on all data up to origin; return 21-business-day trajectory."""
    train_df = price_df.loc[:origin][["price"]].reset_index()
    train_df.columns = pd.Index(["ds", "y"])

    model = Prophet(
        interval_width=0.95,
        daily_seasonality=False,
        weekly_seasonality=False,
        yearly_seasonality=True,
        seasonality_mode="multiplicative",
    )
    model.fit(train_df)

    # Predict enough calendar days to cover 21 business days (≈ 31 calendar days)
    future = model.make_future_dataframe(periods=35, freq="D")
    pred   = model.predict(future).set_index("ds")

    bday_dates = pd.bdate_range(start=origin + pd.offsets.BDay(1), periods=21)
    rows = []
    for h, date in enumerate(bday_dates, start=1):
        cal_date = date.normalize()
        if cal_date in pred.index:
            row = pred.loc[cal_date]
        else:
            nearest_idx = int((pred.index - cal_date).abs().argmin())
            row = pred.iloc[nearest_idx]
        rows.append({
            "origin":        origin,
            "forecast_date": date,
            "horizon":       h,
            "yhat":          float(row["yhat"]),
            "yhat_lower":    float(row["yhat_lower"]),
            "yhat_upper":    float(row["yhat_upper"]),
        })

    return pd.DataFrame(rows)


def load_prophet_trajectories(
    price_df: pd.DataFrame,
    origins: list[pd.Timestamp],
    cache_path: Path,
) -> pd.DataFrame:
    """Load from cache or compute full Prophet trajectory for each origin."""
    if cache_path.exists():
        df = pd.read_parquet(cache_path)
        df["origin"]        = pd.to_datetime(df["origin"])
        df["forecast_date"] = pd.to_datetime(df["forecast_date"])
        print(f"Loaded {len(df)} Prophet trajectory rows from cache.")
        return df

    print("Fitting Prophet at 3 origins (~10 s)...")
    frames = []
    for origin in origins:
        print(f"  {origin.date()} ...", end=" ", flush=True)
        frames.append(_fit_prophet_at_origin(price_df, origin))
        print("done")

    df = pd.concat(frames, ignore_index=True)
    df.to_parquet(cache_path, index=False)
    print(f"Saved {len(df)} rows to {cache_path}")
    return df


prophet_traj_df = load_prophet_trajectories(price_df, ORIGINS, PROPHET_TRAJ_CACHE)
prophet_traj_df.head(6)

Loaded 63 Prophet trajectory rows from cache.


,origin,forecast_date,horizon,yhat,yhat_lower,yhat_upper
0,2026-01-05,2026-01-06,1,56.110597,47.564043,63.171764
1,2026-01-05,2026-01-07,2,56.226107,48.657983,64.782960
2,2026-01-05,2026-01-08,3,56.346479,48.503792,64.105882
3,2026-01-05,2026-01-09,4,56.470931,47.860042,64.401306
4,2026-01-05,2026-01-12,5,56.857824,48.616767,64.700857
5,2026-01-05,2026-01-13,6,56.986786,48.967157,65.217871


In [5]:
# ── Prophet trajectories for all 8 shock-experiment origins ──────────────────
# Reuse the same _fit_prophet_at_origin / load_prophet_trajectories helpers
# defined above, but with a separate cache so Act 5 data stays pristine.

PROPHET_SHOCK_TRAJ_CACHE = DATA_DIR / "energy_shock_prophet_trajectories.parquet"

prophet_shock_traj_df = load_prophet_trajectories(price_df, SHOCK_ORIGINS, PROPHET_SHOCK_TRAJ_CACHE)
print(
    f"Shock-origin Prophet trajectories: {len(prophet_shock_traj_df)} rows "
    f"across {prophet_shock_traj_df['origin'].nunique()} origins"
)
prophet_shock_traj_df.groupby("origin")["horizon"].agg(["min", "max", "count"])

Loaded 168 Prophet trajectory rows from cache.
Shock-origin Prophet trajectories: 168 rows across 8 origins


,min,max,count
origin,,,
2026-02-02,1,21,21
2026-02-09,1,21,21
2026-02-17,1,21,21
2026-02-23,1,21,21
2026-03-02,1,21,21
2026-03-09,1,21,21
2026-03-16,1,21,21
2026-03-23,1,21,21


In [6]:
# ── Context snippets — plausibly knowable on each origin date ─────────────────
# These represent the kind of information a professional energy analyst
# would have had access to from public sources: news, vessel-tracking
# services, futures data, and analyst reports published before the origin date.

ORIGIN_CONTEXTS: dict[str, dict] = {
    "2026-01-05": {
        "label": "Jan 5, 2026",
        "threshold_usd": 65.0,
        "context_text": (
            "As of January 5 2026:\n"
            "- WTI crude has been range-bound in the $56–62 band since October 2025 on soft "
            "demand signals and elevated US inventory builds.\n"
            "- OPEC+ is maintaining its current production-cut agreement through Q1 2026; "
            "no rollback has been signalled.\n"
            "- Iranian proxy forces conducted three separate attacks on US logistics assets "
            "in Iraq and Syria in Q4 2025. US-Iran tensions are elevated but have not "
            "escalated to direct military exchange.\n"
            "- Lloyd's of London hull-war insurance premiums for tankers transiting the "
            "Gulf of Oman have risen approximately 15% since September 2025.\n"
            "- The WTI NYMEX forward curve is in mild backwardation: front month $58, "
            "6-month forward approximately $56.\n"
            "- EIA weekly report (Dec 31 2025): US crude inventories 8% below the 5-year "
            "seasonal average."
        ),
    },
    "2026-02-02": {
        "label": "Feb 2, 2026",
        "threshold_usd": 72.0,
        "context_text": (
            "As of February 2 2026:\n"
            "- WTI gained approximately 7% in January, closing near $62, driven by "
            "escalating Persian Gulf tensions.\n"
            "- A US Navy escort mission in the Gulf of Oman was intercepted by Iranian "
            "fast-attack boats on January 28. No shots fired, but the incident was "
            "widely reported and prompted a diplomatic protest from Washington.\n"
            "- OPEC+ called an emergency ministerial consultation for February 10 amid "
            "concerns about supply-chain disruption risk; no production change announced yet.\n"
            "- Goldman Sachs revised its 2026 WTI price target upward to $70–85 in a "
            "February 1 research note, citing a 'geopolitical risk premium re-rating'.\n"
            "- Vessel-tracking data shows tanker transits through the Strait of Hormuz "
            "down approximately 15% week-over-week, as operators seek alternative routings.\n"
            "- Brent/WTI spread widened to $4.50, the largest since early 2024, as "
            "European buyers began bidding up non-Gulf grades.\n"
            "- US intelligence officials stated publicly that Iranian military assets "
            "have been repositioned closer to the Strait of Hormuz."
        ),
    },
    "2026-03-02": {
        "label": "Mar 2, 2026",
        "threshold_usd": 85.0,
        "context_text": (
            "As of March 2 2026:\n"
            "- The US conducted direct airstrikes on Iranian oil-infrastructure targets "
            "on March 1 2026 in response to an Iranian attack on a US carrier group "
            "in the Gulf of Oman on February 26.\n"
            "- Iran declared a partial blockade of the Strait of Hormuz effective "
            "March 1; approximately 20% of global seaborne oil supply transits the Strait.\n"
            "- WTI surged from $62 on February 2 to $71 by March 2 — a 14% move in "
            "one month — and front-month futures gapped up a further $4 at Monday open.\n"
            "- The IEA called an emergency ministerial meeting for March 5 to consider "
            "releasing strategic petroleum reserves.\n"
            "- Saudi Aramco issued force majeure declarations on several customer contracts; "
            "Saudi Arabia activated its emergency supply protocols.\n"
            "- Goldman Sachs issued an updated note on March 1 with a new 2026 WTI target "
            "of $95–115 and flagging a tail-risk scenario of $130 if the blockade persists "
            "beyond 60 days.\n"
            "- WTI NYMEX forward curve has swung into sharp backwardation: front month "
            "$71, 6-month forward $62, signalling market expectation of eventual resolution."
        ),
    },
}

print("Context snippets defined for:", list(ORIGIN_CONTEXTS.keys()))

Context snippets defined for: ['2026-01-05', '2026-02-02', '2026-03-02']


In [7]:
# ── Run LLMP forecasts (or load from cache) ───────────────────────────────────
#
# We use the LLMP module's internals directly so we can supply a pre-compressed
# history DataFrame rather than going through a DataService.  This is appropriate
# for a playground notebook; a production version would use a registered adapter.
#
# Each origin × 2 variants (bare / with-context) = 6 LLM calls.
# Results are cached to data/energy_llmp_context_forecasts.parquet.

from aieng.forecasting.methods.llm_processes.continuous import (
    ContinuousLLMPredictorConfig,
    _build_system_prompt,
    _build_user_prompt,
    _quantiles_per_step,
    _sample_trajectories,
    _stack_trajectories,
)
from aieng.forecasting.methods.llm_processes.base import serialize_history
from aieng.forecasting.evaluation.prediction import STANDARD_QUANTILES
from aieng.forecasting.evaluation.task import ForecastingTask
from aieng.forecasting.data.models import SeriesMetadata


MODEL       = "gemini/gemini-3-flash-preview"
N_SAMPLES   = 20
HORIZON_B   = 21   # ~21 business days ≈ 30 calendar days
PRECISION   = 2

_WTI_TASK = ForecastingTask(
    task_id="wti_crude_30d",
    target_series_id="wti_crude",
    horizons=list(range(1, HORIZON_B + 1)),
    frequency="B",
    description=(
        "WTI crude oil front-month futures price (USD/bbl), "
        "30 trading-day ahead probabilistic forecast. "
        "Forecast the daily closing price for each of the next "
        f"{HORIZON_B} business days."
    ),
)

_WTI_META = SeriesMetadata(
    series_id="wti_crude",
    description="WTI crude oil front-month futures (CL=F, Adj Close)",
    source="Yahoo Finance",
    units="USD/bbl",
    frequency="B",
)


def _run_one_forecast(
    price_df: pd.DataFrame,
    origin: pd.Timestamp,
    context_text: str | None,
    context_tag: str,
) -> dict:
    """Run LLMP at a single origin and return a dict of arrays."""
    history_df = compress_history(price_df, origin)
    history_str = serialize_history(history_df, precision=PRECISION)

    forecast_start = origin + pd.offsets.BDay(1)
    forecast_end   = origin + pd.offsets.BDay(HORIZON_B)

    system_prompt = _build_system_prompt()
    user_prompt   = _build_user_prompt(
        _WTI_TASK, history_str, _WTI_META,
        forecast_start, forecast_end, HORIZON_B,
        context_text=context_text,
    )

    cfg = ContinuousLLMPredictorConfig(
        model=MODEL,
        n_samples=N_SAMPLES,
        temperature=1.0,
        reasoning_effort="disable",
        context_text=context_text,
        context_tag=context_tag,
    )

    parsed, cost_usd, in_tok, out_tok, failures = _sample_trajectories(
        cfg=cfg, system_prompt=system_prompt, user_prompt=user_prompt
    )
    samples = _stack_trajectories([t.values for t in parsed], n_steps=HORIZON_B)
    q_grid  = _quantiles_per_step(samples)   # (HORIZON_B, len(STANDARD_QUANTILES))

    dates = pd.bdate_range(start=origin + pd.offsets.BDay(1), periods=HORIZON_B)

    rows = []
    for h_idx in range(HORIZON_B):
        row: dict = {
            "origin":      origin,
            "context_tag": context_tag,
            "forecast_date": dates[h_idx],
            "horizon":     h_idx + 1,
            "median":      float(q_grid[h_idx, STANDARD_QUANTILES.index(0.50)]),
            "cost_usd":    cost_usd,
        }
        for qi, q in enumerate(STANDARD_QUANTILES):
            row[f"q{int(q * 100):02d}"] = float(q_grid[h_idx, qi])
        rows.append(row)

    print(
        f"    origin={origin.date()} tag={context_tag:12s} "
        f"cost=${cost_usd:.4f}  failures={failures}/{N_SAMPLES}"
    )
    return {"rows": rows, "samples": samples.tolist()}


def run_all_forecasts(price_df: pd.DataFrame, cache_path: Path) -> pd.DataFrame:
    """Run (or load) all 6 LLMP forecasts; return a single flat DataFrame."""
    if cache_path.exists():
        df = pd.read_parquet(cache_path)
        df["origin"]        = pd.to_datetime(df["origin"])
        df["forecast_date"] = pd.to_datetime(df["forecast_date"])
        print(f"Loaded {len(df)} LLMP forecast rows from cache.")
        return df

    all_rows: list[dict] = []
    print("Running LLMP forecasts (6 API calls)...")
    for origin in ORIGINS:
        key = origin.strftime("%Y-%m-%d")
        ctx = ORIGIN_CONTEXTS[key]
        for tag, text in [("bare", None), ("context", ctx["context_text"])]:
            result = _run_one_forecast(price_df, origin, text, tag)
            all_rows.extend(result["rows"])

    df = pd.DataFrame(all_rows)
    df.to_parquet(cache_path, index=False)
    print(f"Saved {len(df)} rows to {cache_path}")
    return df


llmp_df = run_all_forecasts(price_df, LLMP_CACHE)
print(f"\nOrigins: {sorted(llmp_df['origin'].dt.date.unique())}")
print(f"Tags:    {sorted(llmp_df['context_tag'].unique())}")
llmp_df.head(6)

Loaded 126 LLMP forecast rows from cache.

Origins: [datetime.date(2026, 1, 5), datetime.date(2026, 2, 2), datetime.date(2026, 3, 2)]
Tags:    ['bare', 'context']


,origin,context_tag,forecast_date,horizon,median,cost_usd,q05,q10,q20,q30,q40,q50,q60,q70,q80,q90,q95
0,2026-01-05,bare,2026-01-06,1,58.450,0.07594,58.1500,58.420,58.450,58.450,58.450,58.450,58.450,58.450,58.624,58.650,58.6595
1,2026-01-05,bare,2026-01-07,2,58.820,0.07594,58.3955,58.410,58.620,58.720,58.780,58.820,58.820,58.820,59.120,59.123,59.1530
2,2026-01-05,bare,2026-01-08,3,59.080,0.07594,58.1935,58.300,58.590,58.769,58.846,59.080,59.150,59.150,59.150,59.180,59.4515
3,2026-01-05,bare,2026-01-09,4,58.835,0.07594,58.0910,58.120,58.394,58.700,58.724,58.835,58.874,59.158,59.226,59.384,59.4315
4,2026-01-05,bare,2026-01-12,5,59.165,0.07594,57.8850,57.937,58.426,58.550,59.052,59.165,59.232,59.323,59.420,59.787,59.8515
5,2026-01-05,bare,2026-01-13,6,59.280,0.07594,57.9150,58.400,58.686,58.959,59.092,59.280,59.368,59.680,59.904,60.120,60.1200


---

## Act 5 — Trajectory: Can the Model See the Move Coming?

Each panel below shows the 30-day forecast fan from one origin date.
Three forecast traces:
- **Grey** — Prophet (statistical baseline; history only)
- **Orange** — LLMP, history only (same information as Prophet, different model family)
- **Green** — LLMP with context (public geopolitical information available on that date)

The **solid blue line** is the realized WTI price (actual outcome).
Shading shows 50% and 90% credible intervals for the LLMP forecasts.

In [8]:
def _add_fan(
    fig: go.Figure,
    dates: "pd.Series",
    lower: "pd.Series",
    upper: "pd.Series",
    median: "pd.Series",
    color: str,
    opacity_band: float,
    name: str,
    show_legend: bool,
    legendgroup: str,
    row: int,
    col: int,
) -> None:
    """Add a forecast fan (CI band + median line) to a subplot panel."""
    # CI shading (fill between lower and upper)
    fig.add_trace(
        go.Scatter(
            x=pd.concat([dates, dates[::-1]]),
            y=pd.concat([lower, upper[::-1]]),
            fill="toself",
            fillcolor=color,
            opacity=opacity_band,
            line=dict(width=0),
            mode="lines",
            showlegend=False,
        ),
        row=row, col=col,
    )
    # Median line
    fig.add_trace(
        go.Scatter(
            x=dates, y=median,
            line=dict(color=color, width=2),
            name=name if show_legend else None,
            showlegend=show_legend,
            legendgroup=legendgroup,
        ),
        row=row, col=col,
    )


def make_trajectory_figure(
    price_df: pd.DataFrame,
    prophet_traj_df: pd.DataFrame,
    llmp_df: pd.DataFrame,
    origins: list[pd.Timestamp],
) -> go.Figure:
    """Three-column subplot: one panel per origin, like-for-like trajectory fan comparison.

    All three methods (Prophet, LLMP-bare, LLMP-with-context) are shown as a
    CI-band + median line over the full 21-business-day forecast horizon, making
    the comparison visually consistent.
    """
    labels = [ORIGIN_CONTEXTS[o.strftime("%Y-%m-%d")]["label"] for o in origins]

    fig = psp.make_subplots(
        rows=1, cols=3,
        subplot_titles=labels,
        shared_yaxes=False,
        horizontal_spacing=0.06,
    )

    for col, origin in enumerate(origins, start=1):
        # ── History (60 days pre-origin) ──────────────────────────────────────
        hist_start = origin - pd.Timedelta(days=60)
        hist = price_df[price_df.index >= hist_start].loc[:origin]
        fig.add_trace(
            go.Scatter(
                x=hist.index, y=hist["price"],
                line=dict(color=CLR_HISTORY, width=2),
                name="History" if col == 1 else None,
                showlegend=(col == 1),
                legendgroup="history",
            ),
            row=1, col=col,
        )

        # ── Actuals post-origin ────────────────────────────────────────────────
        res_date, _ = resolution_price(price_df, origin)
        actuals = price_df[(price_df.index > origin) & (price_df.index <= res_date)]
        fig.add_trace(
            go.Scatter(
                x=actuals.index, y=actuals["price"],
                line=dict(color=CLR_ACTUAL, width=2.5),
                name="Actual" if col == 1 else None,
                showlegend=(col == 1),
                legendgroup="actual",
            ),
            row=1, col=col,
        )

        # ── Prophet trajectory fan ─────────────────────────────────────────────
        pt = prophet_traj_df[prophet_traj_df["origin"] == origin].sort_values("forecast_date")
        if not pt.empty:
            _add_fan(
                fig,
                dates=pt["forecast_date"],
                lower=pt["yhat_lower"],
                upper=pt["yhat_upper"],
                median=pt["yhat"],
                color=CLR_PROPHET,
                opacity_band=0.15,
                name="Prophet 95% CI",
                show_legend=(col == 1),
                legendgroup="prophet",
                row=1, col=col,
            )

        # ── LLMP fans (bare + context) ─────────────────────────────────────────
        for tag, clr, name in [
            ("bare",    CLR_LLMP_BARE, "LLMP — history only"),
            ("context", CLR_LLMP_CTX,  "LLMP — with context"),
        ]:
            sub = llmp_df[(llmp_df["origin"] == origin) & (llmp_df["context_tag"] == tag)].sort_values("forecast_date")
            if sub.empty:
                continue

            # 90% CI (outer, light)
            _add_fan(
                fig,
                dates=sub["forecast_date"],
                lower=sub["q05"],
                upper=sub["q95"],
                median=sub["median"] if "median" in sub.columns else sub["q50"],
                color=clr,
                opacity_band=0.10,
                name=name,
                show_legend=(col == 1),
                legendgroup=f"llmp_{tag}",
                row=1, col=col,
            )
            # 50% CI (inner, darker)
            if "q25" in sub.columns and "q75" in sub.columns:
                fig.add_trace(
                    go.Scatter(
                        x=pd.concat([sub["forecast_date"], sub["forecast_date"][::-1]]),
                        y=pd.concat([sub["q25"], sub["q75"][::-1]]),
                        fill="toself",
                        fillcolor=clr,
                        opacity=0.20,
                        line=dict(width=0),
                        mode="lines",
                        showlegend=False,
                    ),
                    row=1, col=col,
                )

        # ── Origin marker ──────────────────────────────────────────────────────
        fig.add_vline(
            x=origin.timestamp() * 1000,
            line=dict(color="#888", dash="dash", width=1),
            row=1, col=col,
        )

    fig.update_layout(
        title=dict(
            text="30-Day WTI Forecast Trajectories — Prophet vs. LLMP vs. LLMP + Context",
            font=dict(size=15),
        ),
        height=420,
        width=1200,
        legend=dict(orientation="h", y=-0.18),
        template="plotly_white",
        margin=dict(t=60, b=90),
    )
    fig.update_yaxes(title_text="WTI (USD/bbl)", col=1)
    return fig


make_trajectory_figure(price_df, prophet_traj_df, llmp_df, ORIGINS).show()

The January origin stays within Prophet's CI. February and March do not — those are the
cases where context makes the difference. Prophet's 30-day median ends up ~$61 for all three
regardless of what's happening in the world. The LLMP with context shifts upward when the
news warrants it.

---

## Act 6 — Binary Forecasting: Will WTI Rise by More Than $5 This Week?

Act 5 showed that trajectory forecasts diverge sharply once context enters the picture.
Here we pin it to a single, evaluable binary question:

> **P(WTI day-5 close > today's price + $5/bbl)** — 8 weekly origins, Feb 2 – Mar 23, 2026

**Why Prophet fails here structurally:** Prophet always forecasts reversion toward its
long-run trend (~$63). When WTI is at $71–$95, Prophet's implied probability that prices
rise *further* by $5+ is essentially zero. But they did — by $23 on Mar 2 and $15 on Mar 23.
The Analyst Agent reads the news and disagrees.

| Origin | WTI | 5-day Δ | Outcome |
|---|---|---|---|
| Feb 2 | $62.14 | +$2.22 | No |
| Feb 9 | $64.36 | −$2.03 | No |
| Feb 17 | $62.33 | +$3.30 | No |
| Feb 23 | $66.31 | +$4.92 | No |
| Mar 2 | $71.23 | **+$23.54** | **Yes — upward shock** |
| Mar 9 | $94.77 | −$1.27 | No |
| Mar 16 | $93.50 | −$5.37 | No |
| Mar 23 | $88.13 | **+$14.75** | **Yes — upward shock** |

In [9]:
import asyncio
import json
import re

import scipy.interpolate
import scipy.stats
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types as genai_types
from google.genai.types import GenerateContentConfig
import litellm

# ── Parameters ────────────────────────────────────────────────────────────────
SHOCK_THRESHOLD      = 5.0   # day-5 close > today + $5 = "upward shock"
SHOCK_HORIZON        = 5     # business days (1 trading week)
SHOCK_ANALYST_CACHE  = DATA_DIR / "energy_upshock_analyst_forecasts.json"
SHOCK_CONTEXT_CACHE  = DATA_DIR / "energy_upshock_news_context.json"

# ── Ground truth ──────────────────────────────────────────────────────────────
def check_shock_outcome(
    price_df: pd.DataFrame,
    origin: pd.Timestamp,
    threshold: float,
    horizon_bdays: int,
) -> tuple[int, float]:
    """Return (outcome, delta): outcome=1 if day-H close > origin_price + threshold."""
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    future_days  = price_df[price_df.index > origin].iloc[:horizon_bdays]
    delta        = float(future_days.iloc[-1]["price"]) - origin_price
    return (1 if delta > threshold else 0), delta

# ── Prophet P(upward shock) ───────────────────────────────────────────────────
def prophet_prob_shock(
    prophet_traj_sub: pd.DataFrame,
    origin_price: float,
    threshold: float,
    horizon: int = SHOCK_HORIZON,
) -> float:
    """P(price_h > origin + threshold) from Prophet's 95% CI (Gaussian approx)."""
    row = prophet_traj_sub[prophet_traj_sub["horizon"] == horizon]
    if row.empty:
        return float("nan")
    row = row.iloc[0]
    sigma = (float(row["yhat_upper"]) - float(row["yhat_lower"])) / (2 * 1.96)
    if sigma <= 0:
        return 1.0 if float(row["yhat"]) > origin_price + threshold else 0.0
    return float(np.clip(
        1.0 - scipy.stats.norm.cdf(origin_price + threshold, loc=float(row["yhat"]), scale=sigma),
        0.0, 1.0,
    ))

# ── Context Agent (Google ADK + google_search) ────────────────────────────────
_CONTEXT_SYSTEM = """\
You are an oil market intelligence specialist with access to web search.

CRITICAL TEMPORAL CONSTRAINT — you are simulating the perspective of an analyst
as of {cutoff_date}.
- Include ONLY information that was publicly available BEFORE {cutoff_date}.
- EXCLUDE any events, market moves, or data from {cutoff_date} or later.
- If a search result appears to post-date the cutoff, skip it entirely.
- This constraint is absolute. Violating it would corrupt the forecast.

Search for and summarise oil-market-relevant information focused on:
- WTI/Brent crude price level and recent trend
- OPEC+ production decisions and supply outlook
- Geopolitical risks in the Persian Gulf, Middle East, key shipping lanes
- US Strategic Petroleum Reserve and energy policy signals
- Notable tanker/shipping incidents or supply chain disruption signals
- Any published analyst forecasts or unusual price-target revisions

Focus especially on factors that could cause a *sudden large move* in WTI crude
over the next 5–10 days. Return a concise structured markdown summary (3–5 paragraphs).\
"""

async def _retrieve_oil_context_async(cutoff_date: str) -> str:
    """Run the Context Agent and return its oil-market summary as of cutoff_date."""
    _APP = "oil-context-agent"
    session_svc = InMemorySessionService()
    agent = LlmAgent(
        name="oil_context_agent",
        instruction=_CONTEXT_SYSTEM.format(cutoff_date=cutoff_date),
        tools=[google_search],
        model="gemini-3-flash-preview",
        generate_content_config=GenerateContentConfig(temperature=0.1, max_output_tokens=2048),
    )
    runner = Runner(agent=agent, app_name=_APP, session_service=session_svc)
    session = await session_svc.create_session(app_name=_APP, user_id="nb")
    prompt = (
        f"Provide an oil market intelligence briefing as of {cutoff_date}. "
        f"Focus on supply risks, OPEC+ policy, Persian Gulf geopolitics, and any factors "
        f"that could cause a sudden large price move in WTI crude in the next 5–10 days. "
        f"IMPORTANT: only use information available before {cutoff_date}."
    )
    content = genai_types.Content(role="user", parts=[genai_types.Part(text=prompt)])
    async for event in runner.run_async(user_id="nb", session_id=session.id, new_message=content):
        if event.is_final_response() and event.content and event.content.parts:
            return event.content.parts[0].text or ""
    return ""

# ── Analyst Agent (litellm / Gemini Flash) ────────────────────────────────────
_ANALYST_SYSTEM = """\
You are an expert oil market analyst making short-term probabilistic forecasts.

You will receive:
  1. Recent WTI crude oil price history (daily, most recent last)
  2. An oil market intelligence briefing with a strict temporal cutoff

Your task: estimate P(up) — the probability that WTI will close MORE THAN
${threshold}/bbl HIGHER than today's price at the end of {horizon} trading days.

This is a directional upside question only.

Be calibrated:
- No unusual upside catalyst → base rate ~10-15%.
- Escalating but unconfirmed geopolitical risk → 20-40%.
- Confirmed major supply disruption actively driving prices higher → 60-85%.

Respond ONLY in valid JSON with exactly these fields:
{{
  "probability_up": <float between 0 and 1>,
  "direction_bias": "<up|down|neutral>",
  "reasoning": "<2-4 sentences>",
  "key_signals": ["<signal 1>", "<signal 2>", "<signal 3>"],
  "confidence": "<high|medium|low>"
}}
Output ONLY the JSON object. No other text.\
"""

def _run_analyst(
    history_str: str,
    news_context: str,
    origin_date: str,
    origin_price: float,
) -> dict:
    """Analyst Agent: reason from price history + news context → structured dict."""
    system = _ANALYST_SYSTEM.format(horizon=SHOCK_HORIZON, threshold=int(SHOCK_THRESHOLD))
    target = origin_price + SHOCK_THRESHOLD
    user_prompt = (
        f"### WTI Price History (ending {origin_date})\n\n"
        f"{history_str}\n\n"
        f"---\n"
        f"### Oil Market Briefing (as of {origin_date})\n\n"
        f"{news_context}\n\n"
        f"---\n"
        f"Current WTI price: ${origin_price:.2f}/bbl on {origin_date}.\n"
        f"Estimate P(WTI closes above ${target:.2f} in {SHOCK_HORIZON} trading days)."
    )
    response = litellm.completion(
        model="gemini/gemini-3-flash-preview",
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user_prompt}],
        temperature=0.2,
        response_format={"type": "json_object"},
    )
    raw = response.choices[0].message.content or "{}"
    raw = re.sub(r"^```(?:json)?\s*", "", raw.strip())
    raw = re.sub(r"\s*```$", "", raw)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r'"probability_up"\s*:\s*([\d.]+)', raw)
        prob = float(m.group(1)) if m else float("nan")
        return {"reasoning": raw[:500], "probability_up": prob,
                "key_signals": [], "direction_bias": "?", "confidence": "?"}

print("Agent definitions loaded.")
print(f"  SHOCK_THRESHOLD : +${SHOCK_THRESHOLD:.0f}/bbl  |  SHOCK_HORIZON : {SHOCK_HORIZON} bdays")
print(f"  Question        : P(WTI day-5 close > today + ${SHOCK_THRESHOLD:.0f})")
print(f"  Context Agent   : Google ADK + google_search  (gemini-3-flash-preview, temporal cutoff)")
print(f"  Analyst Agent   : gemini-3-flash-preview  →  JSON  (probability_up + reasoning)")

Agent definitions loaded.
  SHOCK_THRESHOLD : +$5/bbl  |  SHOCK_HORIZON : 5 bdays
  Question        : P(WTI day-5 close > today + $5)
  Context Agent   : Google ADK + google_search  (gemini-3-flash-preview, temporal cutoff)
  Analyst Agent   : gemini-3-flash-preview  →  JSON  (probability_up + reasoning)


In [10]:
# ── Run pipeline (or load from cache) ────────────────────────────────────────
# For each of 8 weekly shock-experiment origins:
#   (1) Context Agent fetches news; (2) Analyst Agent estimates P(shock).
# Top-level `await` works in Jupyter — uses the kernel's running event loop.

if SHOCK_ANALYST_CACHE.exists() and SHOCK_CONTEXT_CACHE.exists():
    with open(SHOCK_ANALYST_CACHE) as f:
        shock_analyst_results: list[dict] = json.load(f)
    with open(SHOCK_CONTEXT_CACHE) as f:
        shock_news_contexts: dict[str, str] = json.load(f)
    print(f"Loaded {len(shock_analyst_results)} cached shock-experiment forecasts.")
else:
    shock_news_contexts = {}
    shock_analyst_results = []

    for origin in SHOCK_ORIGINS:
        key          = origin.strftime("%Y-%m-%d")
        cutoff       = (origin - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])

        print(f"\n{'─' * 60}")
        print(f"Origin {key}  (context cutoff: {cutoff})  WTI ${origin_price:.2f}")

        print("  [1/2] Context Agent — searching for oil market intelligence ...")
        ctx = await _retrieve_oil_context_async(cutoff)
        shock_news_contexts[key] = ctx
        print(f"        {len(ctx):,} chars retrieved")

        print(f"  [2/2] Analyst Agent — reasoning about P(shock > ${SHOCK_THRESHOLD:.0f}) ...")
        hist     = compress_history(price_df, origin)
        hist_str = serialize_history(hist, precision=2)
        result   = _run_analyst(hist_str, ctx, key, origin_price)
        result["origin"] = key
        shock_analyst_results.append(result)
        p_up = result.get("probability_up", float("nan"))
        p_str = f"{p_up:.2f}" if isinstance(p_up, float) else str(p_up)
        print(f"        P(up>+${SHOCK_THRESHOLD:.0f})={p_str}  confidence={result.get('confidence', '?')}")

    with open(SHOCK_ANALYST_CACHE, "w") as f:
        json.dump(shock_analyst_results, f, indent=2)
    with open(SHOCK_CONTEXT_CACHE, "w") as f:
        json.dump(shock_news_contexts, f, indent=2)
    print("\nSaved to cache.")

# ── Assemble binary_df: Prophet + Analyst Agent + Always-50% baseline ─────────
shock_analyst_by_origin = {r["origin"]: r for r in shock_analyst_results}

binary_rows = []
for origin in SHOCK_ORIGINS:
    key          = origin.strftime("%Y-%m-%d")
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    outcome, max_move = check_shock_outcome(price_df, origin, SHOCK_THRESHOLD, SHOCK_HORIZON)

    # Prophet: P(shock) from Gaussian approximation to 5-day trajectory CI
    pt_sub = prophet_shock_traj_df[prophet_shock_traj_df["origin"] == origin]
    p_prob = prophet_prob_shock(pt_sub, origin_price, SHOCK_THRESHOLD)

    # Analyst Agent
    a      = shock_analyst_by_origin.get(key, {})
    a_prob = float(a.get("probability_up", float("nan")))

    for method, prob in [("Prophet", p_prob), ("Analyst Agent", a_prob), ("Always 50%", 0.5)]:
        binary_rows.append({
            "origin": key,
            "origin_price": origin_price,
            "max_move": max_move,
            "outcome": outcome,
            "method": method,
            "prob": prob,
            "brier": (prob - outcome) ** 2,
            "reasoning": a.get("reasoning") if method == "Analyst Agent" else None,
            "key_signals": a.get("key_signals", []) if method == "Analyst Agent" else [],
            "confidence": a.get("confidence") if method == "Analyst Agent" else None,
            "direction_bias": a.get("direction_bias") if method == "Analyst Agent" else None,
        })

binary_df = pd.DataFrame(binary_rows)
binary_df["origin_dt"] = pd.to_datetime(binary_df["origin"])
binary_df = binary_df.sort_values(["origin_dt", "method"]).reset_index(drop=True)

# ── Quick sanity check ─────────────────────────────────────────────────────────
print("\n── Binary forecast summary ─────────────────────────────────────────────")
summary = binary_df.pivot_table(
    index="origin", columns="method", values=["prob", "brier"], aggfunc="first"
)
print(summary.to_string())

Loaded 8 cached shock-experiment forecasts.

── Binary forecast summary ─────────────────────────────────────────────
                brier                                   prob                            
method     Always 50% Analyst Agent       Prophet Always 50% Analyst Agent       Prophet
origin                                                                                  
2026-02-02       0.25        0.0400  6.582261e-04        0.5          0.20  2.565592e-02
2026-02-09       0.25        0.1024  3.039773e-04        0.5          0.32  1.743494e-02
2026-02-17       0.25        0.0144  8.373381e-03        0.5          0.12  9.150618e-02
2026-02-23       0.25        0.0324  1.198964e-03        0.5          0.18  3.462606e-02
2026-03-02       0.25        0.0225  9.964893e-01        0.5          0.85  1.756918e-03
2026-03-09       0.25        0.1600  1.492722e-27        0.5          0.40  3.863576e-14
2026-03-16       0.25        0.4225  7.371441e-21        0.5          0.65  8.585

In [11]:
# ── Display: news context + agent reasoning for all 8 shock origins ───────────
from IPython.display import display, Markdown as MD

for origin in SHOCK_ORIGINS:
    key          = origin.strftime("%Y-%m-%d")
    a            = shock_analyst_by_origin.get(key, {})
    ctx          = shock_news_contexts.get(key, "")
    origin_price = float(price_df[price_df.index >= origin].iloc[0]["price"])
    outcome, max_move = check_shock_outcome(price_df, origin, SHOCK_THRESHOLD, SHOCK_HORIZON)

    outcome_icon = "🔴 SHOCK" if outcome else "🟢 Calm"
    a_prob = float(a.get("probability_up", float("nan")))
    bs     = (a_prob - outcome) ** 2
    delta  = float(price_df[price_df.index > origin].iloc[:SHOCK_HORIZON].iloc[-1]["price"]) - origin_price

    md = (
        f"---\n"
        f"### {key} — WTI ${origin_price:.2f}/bbl  ·  {outcome_icon}"
        f"  (5-day Δ: {delta:+.2f}, threshold +${SHOCK_THRESHOLD:.0f})\n\n"
        f"<details><summary>📰 Oil market context (cutoff: "
        f"{(origin - pd.Timedelta(days=1)).strftime('%Y-%m-%d')}) — click to expand</summary>\n\n"
        f"{ctx}\n\n</details>\n\n"
        f"**Reasoning:** {a.get('reasoning', 'N/A')}  \n"
        f"**Key signals:** {', '.join(a.get('key_signals', []))}  \n"
        f"**Bias:** {a.get('direction_bias', '?')}  "
        f"**Confidence:** {a.get('confidence', '?')}  "
        f"**P(up > +${SHOCK_THRESHOLD:.0f}) = {a_prob:.0%}**  "
        f"**Brier:** {bs:.3f}"
    )
    display(MD(md))

---
### 2026-02-02 — WTI $62.14/bbl  ·  🟢 Calm  (5-day Δ: +2.22, threshold +$5)

<details><summary>📰 Oil market context (cutoff: 2026-02-01) — click to expand</summary>

**Oil Market Intelligence Briefing**
**Date:** February 1, 2026
**Analyst:** oil_context_agent

### **Executive Summary: Market at a Geopolitical Crossroads**
As of February 1, 2026, WTI crude is trading near **$67.00/bbl**, having experienced a volatile January characterized by a $10/bbl "geopolitical premium" spike. While long-term fundamentals remain bearish due to a projected global supply surplus of 2.1–4.0 million barrels per day (mb/d) for 2026, the immediate term is dominated by a "kinetic risk" premium. Market participants are hyper-focused on the escalating tensions between the United States and Iran, which have pushed Brent crude toward the **$70/bbl** mark. The primary concern for the next 5–10 days is a sudden, large upward move in WTI should these tensions transition from rhetoric to active military engagement or shipping lane blockades.

### **OPEC+ Policy and Supply Outlook**
OPEC+ has entered 2026 with a "strategic pause," reconfirming in early January that the eight-nation coalition (led by Saudi Arabia and Russia) will maintain current production quotas through the end of Q1. This decision aims to defend a price floor in the low $60s for Brent against a backdrop of surging non-OPEC+ production from the U.S., Brazil, and Guyana. However, internal friction is visible; analysts note that every additional cut risks ceding market share to U.S. shale, which remains functional at $55–$60 WTI. The market is currently pricing in a "mathematical discrepancy" where projected demand of 25.6 mb/d from OPEC is dwarfed by their current capacity, suggesting that any sign of quota cheating or a surprise production increase could trigger a sharp $5–$10 drop in WTI.

### **Geopolitical Risks and Shipping Disruptions**
The Persian Gulf is currently the highest-risk variable for a sudden price spike. Throughout January, the U.S. and Israel have signaled increasing readiness for military action against Iranian infrastructure. Shipping insurance premiums for the Strait of Hormuz—which carries roughly 2

</details>

**Reasoning:** While geopolitical tensions between the US and Iran provide a significant risk premium, the underlying market fundamentals for 2026 are heavily bearish due to a projected 2.1–4.0 mb/d supply surplus. The recent price drop from $65 to $62 suggests the market is currently fading the immediate risk of escalation, making a $5 recovery in just five trading days unlikely without a confirmed military event. Consequently, the path of least resistance remains downward toward the fundamental floor unless a major supply disruption occurs.  
**Key signals:** Strait of Hormuz shipping insurance spikes, OPEC+ quota compliance levels, US-Iran military posturing  
**Bias:** down  **Confidence:** medium  **P(up > +$5) = 20%**  **Brier:** 0.040

---
### 2026-02-09 — WTI $64.36/bbl  ·  🟢 Calm  (5-day Δ: -2.03, threshold +$5)

<details><summary>📰 Oil market context (cutoff: 2026-02-08) — click to expand</summary>

As of February 8, 2026, WTI crude is trading in a range of **$68–$74 per barrel**, characterized by extreme volatility as the market balances a projected 2026 supply surplus against a rapidly deteriorating geopolitical landscape. While the EIA’s January outlook forecasted a bearish annual average of $52/b for WTI due to rising production in the "Americas quintet" (US, Canada, Brazil, Guyana, and Argentina), prices have recently trended upward. This "geopolitical floor" is driven by the fallout from late-2025 Israeli strikes on Lebanon and subsequent Iranian threats to maritime traffic, which have kept a significant risk premium embedded in front-month contracts.

OPEC+ remains in a defensive posture, having recently reaffirmed its commitment to voluntary production cuts through the first half of 2026. Internal cohesion is under pressure; as of January, the alliance is grappling with "compensation cuts" from Iraq and the UAE, which have effectively neutralized planned production increases. Market participants are closely watching for any signs of the UAE’s long-rumored pivot toward a more independent production strategy, especially as Abu Dhabi nears its 5 million bpd capacity target. For now, the alliance’s "cautious approach" is the primary counterweight to the 2.5 million bpd in non-OPEC supply growth expected this year.

The most critical risk for a **sudden large move (+$15–20/b)** in the next 5–10 days is the potential for a "de facto" closure of the Strait of Hormuz. Following the ousting of Venezuelan President Nicolás Maduro on January 3, 2026, and the subsequent instability in Caracas, global spare capacity is perceived as thinner than official figures suggest. Intelligence reports indicate that Iran has moved mobile missile batteries near the Persian Gulf, and any kinetic incident involving a commercial tanker or a US naval vessel would likely trigger an immediate spike toward $100/b. Conversely, a sudden diplomatic de-escalation or a "ceasefire deal" in the Levant could see WTI collapse back toward the $60 level as the market refocuses on the 4.5 million bpd "supply glut" projected by the IEA for Q2 2026.

In the US, the Trump administration has prioritized refilling the Strategic Petroleum Reserve (SPR), which currently stands at approximately **411 million barrels** (roughly 58

</details>

**Reasoning:** WTI is currently trading at $64.36, which is below the $68-$74 range cited in the market briefing, suggesting a potential upward correction to align with the perceived 'geopolitical floor.' Escalating Iranian military posturing near the Strait of Hormuz and the ousting of the Venezuelan president have introduced a significant risk premium that could easily trigger a $5 spike on any kinetic news. While long-term supply forecasts are bearish, OPEC+ production discipline and US SPR refilling provide strong short-term support.  
**Key signals:** Iranian missile battery deployment near Hormuz, OPEC+ voluntary cut compliance, Venezuelan political instability  
**Bias:** up  **Confidence:** medium  **P(up > +$5) = 32%**  **Brier:** 0.102

---
### 2026-02-17 — WTI $62.33/bbl  ·  🟢 Calm  (5-day Δ: +3.30, threshold +$5)

<details><summary>📰 Oil market context (cutoff: 2026-02-16) — click to expand</summary>

### **Oil Market Intelligence Briefing**
**Date:** February 16, 2026
**Subject:** Q1 Supply Risks and Geopolitical Volatility Outlook

#### **Price Trend and Market Sentiment**
As of mid-February 2026, WTI crude is trading in the **$63–$66 range**, showing signs of a fragile recovery after drifting

</details>

**Reasoning:** A $5 move represents an 8% increase from the current price of $62.33, a threshold rarely met without a major supply-side shock. Recent price action shows a downward drift from $64.63 to $62.33 over the last four trading sessions, indicating a lack of immediate upside momentum. While the briefing mentions geopolitical risks, the market is currently characterizing the recovery as fragile rather than explosive.  
**Key signals:** Recent downward price drift, Fragile market recovery sentiment, High 8% volatility threshold  
**Bias:** neutral  **Confidence:** high  **P(up > +$5) = 12%**  **Brier:** 0.014

---
### 2026-02-23 — WTI $66.31/bbl  ·  🟢 Calm  (5-day Δ: +4.92, threshold +$5)

<details><summary>📰 Oil market context (cutoff: 2026-02-22) — click to expand</summary>

**Oil Market Intelligence Briefing**
**Date:** February 22, 2026
**Status:** Confidential / Market Intelligence

### **Price Level and Recent Trend**
As of late February 2026, WTI crude is trading in the **$66–$69 range**, having established a modest upward trend from the $64 average seen in late 2025. While global fundamentals suggest a looming supply surplus for the second half of the year, prices are currently "bloated" by a **geopolitical risk premium estimated at $4–$10 per barrel**. This premium is largely driven by aggressive rhetoric following the recent U.S. State of the Union address, which signaled a harder stance on Middle Eastern adversaries. Market volatility has increased over the last 48 hours as traders weigh soft physical demand against the potential for a sudden security escalation.

### **OPEC+ Policy and Supply Outlook**
OPEC+ remains in a "wait-and-see" posture after the eight leading producers (including Saudi Arabia, Russia, and the UAE) reaffirmed their decision to **pause production increases through March 2026**. This freeze on the previously planned unwinding of voluntary cuts has tightened the Q1 balance more than analysts expected in December. Meanwhile, U.S. domestic production is showing signs of a **plateau at approximately 13.6 million bpd**, with the Permian Basin facing increased scrutiny as drilling productivity gains begin to diminish. The combination of OPEC+ restraint and stagnant U.S. growth has removed the immediate threat of a supply glut, though a scheduled JMMC monitoring meeting in early March is the next major policy milestone.

### **Geopolitical Risks and Shipping Lanes**
The **Persian Gulf** has returned to the center of market anxiety. Intelligence reports and recent diplomatic friction suggest that Iran is currently the "loudest risk" to global supply. While the Strait of Hormuz remains open, there is heightened concern regarding military posturing and the potential for new, more stringent sanctions that could sideline up to 1 million bpd of "shadow" exports. Any disruption to the **Bab el-Mandeb or the Strait of Hormuz** would be catastrophic, as global commercial refined product stocks have already thinned to roughly 45–47 days of demand. Shipping insurance premiums for tankers in the region have ticked upward this week, a reliable leading indicator of perceived physical risk.

### **US Policy and SPR Status**
The U.S. Strategic Petroleum Reserve (SPR) currently stands at approximately **411 million barrels**. The administration has shifted to an aggressive refill priority, signaling a "floor" for WTI by issuing standing solicitations to purchase crude whenever prices dip below **$75–$80/

</details>

**Reasoning:** A $5 increase in five trading days requires a transition from geopolitical rhetoric to a tangible supply disruption or an aggressive acceleration of SPR refill purchases. While the Persian Gulf tensions and the stated $75-$80 SPR floor provide strong upward support, the current 'soft' physical demand and the fact that prices have remained well below the SPR floor for months suggest a high threshold for such a rapid price spike.  
**Key signals:** Rising shipping insurance premiums in the Persian Gulf, US SPR refill solicitation activity and volume, OPEC+ production freeze through March 2026  
**Bias:** up  **Confidence:** medium  **P(up > +$5) = 18%**  **Brier:** 0.032

---
### 2026-03-02 — WTI $71.23/bbl  ·  🔴 SHOCK  (5-day Δ: +23.54, threshold +$5)

<details><summary>📰 Oil market context (cutoff: 2026-03-01) — click to expand</summary>

### **Oil Market Intelligence Briefing: March 1, 2026**

**Current Price Action and Immediate Trend**
As of the market open on March 1, 2026, WTI crude is undergoing an explosive upward correction. After closing at approximately **$67.00/bbl on February 27**, prices have gapped significantly higher in early trading following the weekend’s catastrophic geopolitical escalations. Brent crude has already breached the **$100/bbl** mark. The market is currently in a state of "price discovery" as traders attempt to quantify the impact of a near-total cessation of seaborne exports from the Persian Gulf.

**Geopolitical Catalyst: The "Dual-Chokepoint" Crisis**
The primary driver for a sudden, large move in the next 5–10 days is the **de facto closure of the Strait of Hormuz**, which occurred following US and Israeli strikes on Iranian military and nuclear infrastructure on **February 28, 2026**. Iran has responded by effectively shutting down commercial traffic through the Strait, a corridor that handles roughly 20% of global oil supply (11–16 million bpd). Simultaneously, Houthi forces in Yemen resumed large-scale attacks on Red Sea shipping on February 28, reversing the stability seen since the late 2025 ceasefire. For the first time in modern history, both the Strait of Hormuz and the Bab el-Mandeb are restricted, leaving roughly three-quarters of Gulf supplies with no viable maritime exit.

**OPEC+ Policy and Supply Outlook**
The OPEC+ ministerial meeting is scheduled for today, **March 1, 2026**. Prior to this weekend, the alliance (led by Saudi Arabia and Russia) had reaffirmed a decision to **pause production increases** through Q1 2026 due to seasonal demand weakness. However, the sudden removal of over 10 million bpd of regional production capacity—including significant volumes from Saudi Arabia, the UAE, and Kuwait that are now "trapped" behind the Hormuz blockade—has rendered previous quotas obsolete. Market participants are watching for an emergency declaration from the alliance, though spare capacity outside the conflict zone (primarily in Russia and West Africa) is insufficient to offset a prolonged Gulf outage.

**US Policy and Strategic Petroleum Reserve (SPR)**
The Trump administration, which took office in early 2025, has made refilling the SPR a "Department-level priority," bringing levels to approximately **415.4 million barrels** as of mid-February 2026. While the administration had been a net buyer of crude to "fill the caverns to the top," the current crisis has shifted the focus toward an emergency release. Analysts expect a massive, coordinated IEA release to be announced within days to provide a bridge for global refiners. However, the structural ceiling of US export infrastructure and the "exchange" (loan) model favored by the current Department of Energy may limit the immediate cooling effect on WTI prices.

**5–10 Day Risk Summary**
The risk of a **$20–$30/bbl spike in WTI** over the next 10 days is extreme. If the Strait of Hormuz remains unnavigable and insurance providers continue to cancel war-risk coverage for Gulf-transiting vessels, WTI is forecast to test **$95–$100/bbl** by mid-March. The primary "downside" risk to this rally would be an immediate, credible de-escalation or a successful US-led military escort program for tankers, though neither appears imminent as of March 1. Traders should brace for historic volatility and potential "limit up" trading sessions.

</details>

**Reasoning:** The de facto closure of the Strait of Hormuz and the Bab el-Mandeb represents a catastrophic supply shock, potentially removing 10-16 million bpd from the global market. With Brent already trading above $100/bbl, WTI is positioned for a massive catch-up rally as traders price in the near-total cessation of Persian Gulf seaborne exports. A $5 move represents only a ~7% increase, which is conservative given the briefing's forecast of WTI testing $95-$100/bbl in the coming days.  
**Key signals:** Closure of the Strait of Hormuz, Brent crude breaching $100/bbl, Simultaneous Houthi attacks in the Red Sea  
**Bias:** up  **Confidence:** high  **P(up > +$5) = 85%**  **Brier:** 0.023

---
### 2026-03-09 — WTI $94.77/bbl  ·  🟢 Calm  (5-day Δ: -1.27, threshold +$5)

<details><summary>📰 Oil market context (cutoff: 2026-03-08) — click to expand</summary>

**Oil Market Intelligence Briefing**
**Date:** March 8, 2026
**Subject:** Supply Risks and Volatility Outlook Following Middle East Escalation

### **Price Action and Market Sentiment**
WTI crude is currently experiencing extreme upward volatility, trading in the **$88–$92/bbl range** as of early March 8. This represents a

</details>

**Reasoning:** WTI has entered a parabolic phase, surging over $23/bbl in the last week due to escalating Middle East geopolitical risks. While the momentum is exceptionally strong, the market is approaching the $100 psychological resistance level, which may trigger profit-taking or stabilization. A further $5 increase in just five days requires the current 'risk' to transition into a confirmed, large-scale physical supply disruption.  
**Key signals:** Middle East geopolitical escalation, $100 psychological resistance level, Extreme parabolic price momentum  
**Bias:** up  **Confidence:** medium  **P(up > +$5) = 40%**  **Brier:** 0.160

---
### 2026-03-16 — WTI $93.50/bbl  ·  🟢 Calm  (5-day Δ: -5.37, threshold +$5)

<details><summary>📰 Oil market context (cutoff: 2026-03-15) — click to expand</summary>

**Oil Market Intelligence Briefing: March 15, 2026**

**Price Action and Market Trend**
WTI crude is currently experiencing extreme volatility, trading near **$96–$99 per barrel** after a vertical ascent from the $67 level in late February. This 45%+ surge in just over two weeks is driven by the "Hormuz Shock"—the effective closure of the Strait of Hormuz to commercial traffic following the commencement of "Operation Epic Fury" (the US-Israeli aerial campaign against Iranian infrastructure). While the market saw a brief $5–$8 "relief dip" following the March 11 announcement of a massive global emergency reserve release, prices remain pinned near three-year highs as the physical reality of a 14 million barrel per day (bpd) supply deficit outweighs paper-market interventions.

**Geopolitical Crisis and Supply Disruption**
The Persian Gulf is currently a "no-go" zone for international tankers. Following the February 28 blockade, Iranian retaliatory strikes have reportedly damaged production facilities in neighboring Gulf states, leading to an unprecedented **14 million bpd of shut-in capacity**. The IEA’s March 12 report characterizes this as the largest supply disruption in history, surpassing the 1973 embargo. Shipping giants Maersk and CMA CGM have indefinitely suspended all bookings in the region, and insurance premiums for the few remaining vessels in the Gulf of Oman have reached prohibitive levels.

**OPEC+ and Strategic Policy Responses**
OPEC+ cohesion is under severe strain; member production plunged by an estimated 27% in March as Gulf producers lost export outlets. In a historic move to stabilize the global economy, the IEA coordinated a **400-million-barrel emergency release** on March 11, with the Trump administration committing **172 million barrels** from the US Strategic Petroleum Reserve (SPR). This drawdown will push US SPR levels to their lowest point since 1983, signaling that Western powers are "all-in" on using strategic stocks to bridge the gap until the Strait can be reopened.

**5–10 Day Outlook and Volatility Triggers**
The market is braced for a "sudden large move" in either direction based on the following triggers:
*   **Upside Risk ($110+):** Any confirmed reports of "permanent" damage to Saudi or UAE processing plants (e.g., Abqaiq-style drone strikes) or a failure of the IEA release to reach Asian refineries in time to prevent stock-outs.
*   **Downside Risk (<$85):** Diplomatic "back-channel" signals suggesting a temporary ceasefire or a "humanitarian corridor" for tankers. 
*   **Technical Note:** WTI is currently testing psychological resistance at $100. A clean break above this level could trigger a gamma squeeze in the options market, potentially pushing prices toward $120 within 48 hours. Conversely, any sign of the blockade lifting will likely cause a $15–$20 "crash" as the massive long positions built over the last 10 days liquidate.

</details>

**Reasoning:** The unprecedented 14 million bpd supply deficit caused by the Strait of Hormuz closure remains the dominant fundamental driver, outweighing the temporary psychological relief provided by the IEA's 400-million-barrel reserve release. While the SPR intervention caused a dip to $93.50, the physical reality of shut-in Gulf production and the lack of a diplomatic corridor suggest a high likelihood of prices retesting the $100 psychological resistance. A $5 move is well within the current extreme volatility regime, especially if reports of permanent infrastructure damage surface or the SPR release faces logistical delays.  
**Key signals:** Status of the Strait of Hormuz blockade, Reports of damage to Saudi/UAE processing facilities, Delivery timelines and refinery uptake of IEA emergency stocks  
**Bias:** up  **Confidence:** medium  **P(up > +$5) = 65%**  **Brier:** 0.423

---
### 2026-03-23 — WTI $88.13/bbl  ·  🔴 SHOCK  (5-day Δ: +14.75, threshold +$5)

<details><summary>📰 Oil market context (cutoff: 2026-03-22) — click to expand</summary>

### **Oil Market Intelligence Briefing: March 22, 2026**

**Price Action and Market Sentiment**
WTI crude is currently trading in a volatile range between **$95 and $100 per barrel**, a dramatic surge from the $60–$65 level seen in early February. This "war premium" follows the February 

</details>

**Reasoning:** WTI experienced a sharp $10 decline on March 23, suggesting a potential cooling of the 'war premium' or a significant technical correction from recent highs near $100. While the underlying geopolitical risk remains a potent upside catalyst, the immediate downward momentum must be overcome to achieve a $5 gain within five trading days. A recovery to the $93 level represents a partial retracement of the recent drop, which is plausible if conflict headlines intensify again.  
**Key signals:** Geopolitical escalation/de-escalation news, Technical support levels near $85, Market reaction to the sharp March 23 sell-off  
**Bias:** neutral  **Confidence:** medium  **P(up > +$5) = 35%**  **Brier:** 0.423

In [12]:
# ── Act 6 summary chart ────────────────────────────────────────────────────────
#
# Top panel  — dot+line chart: P(upward shock) for Prophet & Analyst Agent
#              Shock-outcome weeks highlighted with a red column background
#              so the audience can instantly read "model said X, this happened"
#
# Bottom panel — cumulative mean Brier score (lower = better)
#
# Sized narrow (640 px wide) for pptx embedding.

METHOD_COLORS = {"Prophet": CLR_PROPHET, "Analyst Agent": CLR_LLMP_CTX}
METHOD_SYMBOL = {"Prophet": "square",    "Analyst Agent": "circle"}
METHOD_DASH   = {"Prophet": "dot",       "Analyst Agent": "solid"}

origins_ordered = [o.strftime("%Y-%m-%d") for o in SHOCK_ORIGINS]
outcome_by_key  = {
    key: int(binary_df[(binary_df["origin"] == key) & (binary_df["method"] == "Prophet")].iloc[0]["outcome"])
    for key in origins_ordered
}
shock_indices = [i for i, k in enumerate(origins_ordered) if outcome_by_key[k] == 1]

fig = psp.make_subplots(
    rows=2, cols=1,
    row_heights=[0.58, 0.42],
    vertical_spacing=0.20,
    subplot_titles=[
        "P(WTI up > +$5 /bbl in 5 trading days)",
        "Cumulative mean Brier score (lower = better)",
    ],
)

# ── Shock-outcome column shading (both panels) ────────────────────────────────
# Categorical x-axes map category n → integer index n, so use numeric offsets.
for i in shock_indices:
    for row_n, (y0, y1) in [(1, (-0.12, 1.08)), (2, (0.0, 0.30))]:
        fig.add_shape(
            type="rect", layer="below",
            xref=f"x{'' if row_n == 1 else row_n}",
            yref=f"y{'' if row_n == 1 else row_n}",
            x0=i - 0.48, x1=i + 0.48,
            y0=y0, y1=y1,
            fillcolor="rgba(214,39,40,0.12)",
            line_width=0,
        )
    # Label at the top of the shaded column
    fig.add_annotation(
        x=i, y=1.06, text="<b>SHOCK</b>",
        showarrow=False,
        font=dict(size=9, color=CLR_CONFLICT),
        xref="x", yref="y",
    )

# ── Row 1: probability dot + line ─────────────────────────────────────────────
for method in ["Analyst Agent", "Prophet"]:
    sub = binary_df[binary_df["method"] == method].sort_values("origin_dt")
    fig.add_trace(go.Scatter(
        x=sub["origin"],
        y=sub["prob"],
        name=method,
        mode="lines+markers",
        line=dict(color=METHOD_COLORS[method], width=2.5, dash=METHOD_DASH[method]),
        marker=dict(size=10, symbol=METHOD_SYMBOL[method]),
        legendgroup=method,
        showlegend=True,
        hovertemplate="%{x}<br>P(up)=%{y:.0%}<extra>" + method + "</extra>",
    ), row=1, col=1)
    # Probability label above each dot
    for _, r in sub.iterrows():
        fig.add_annotation(
            x=r["origin"], y=r["prob"],
            text=f"{r['prob']:.0%}",
            showarrow=False,
            font=dict(size=8, color=METHOD_COLORS[method]),
            yshift=12,
            xref="x", yref="y",
        )

fig.add_hline(
    y=0.5, line=dict(color="#d0d0d0", dash="dot", width=1.2),
    row=1, col=1,
)
fig.add_annotation(
    x=7, y=0.5, text="50%", showarrow=False,
    font=dict(size=8, color="#bbb"), xshift=22, yref="y",
)

fig.update_yaxes(
    title_text="P(up > +$5)", range=[-0.08, 1.14],
    tickformat=".0%", dtick=0.25, row=1, col=1,
)
fig.update_xaxes(tickangle=-30, row=1, col=1)

# ── Row 2: cumulative mean Brier ───────────────────────────────────────────────
for method in ["Analyst Agent", "Prophet"]:
    sub       = binary_df[binary_df["method"] == method].sort_values("origin_dt")
    cum_brier = sub["brier"].expanding().mean().values
    fig.add_trace(go.Scatter(
        x=sub["origin"].values,
        y=cum_brier,
        name=method,
        mode="lines+markers",
        line=dict(color=METHOD_COLORS[method], width=2.5, dash=METHOD_DASH[method]),
        marker=dict(size=8, symbol=METHOD_SYMBOL[method]),
        legendgroup=method,
        showlegend=False,
        hovertemplate="%{x}<br>Cumul. Brier: %{y:.3f}<extra>" + method + "</extra>",
    ), row=2, col=1)

fig.add_hline(
    y=0.25, line=dict(color="#aaa", dash="dot", width=1.5),
    annotation_text="0.25 random ceiling",
    annotation_position="top right",
    annotation_font=dict(size=9, color="#888"),
    row=2, col=1,
)
fig.update_yaxes(
    title_text="Brier score", range=[0, 0.30],
    row=2, col=1,
)
fig.update_xaxes(tickangle=-30, row=2, col=1)

fig.update_layout(
    title=dict(
        text="Analyst Agent vs. Prophet — Upward Shock (Feb–Mar 2026)",
        x=0.5, font=dict(size=13),
    ),
    height=520, width=640,
    template="plotly_white",
    legend=dict(
        orientation="h", yanchor="bottom", y=1.04, xanchor="right", x=1,
        font=dict(size=11),
    ),
    margin=dict(t=80, b=55, l=60, r=35),
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor="#ececec", gridwidth=0.7)
fig.show()